# NVARC frozen inference: public evaluation and Kaggle submission

This notebook isolates the published NVARC/Qwen3-4B checkpoint's **frozen** inference behavior. It deliberately contains no test-time training, PEFT/LoRA adapter, optimizer, trainer, or gradient-bearing path.

Two modes share the same model and ARC serialization:

- `evaluation`: run the public evaluation set and compare canonical greedy (`G0`), 16-view greedy (`G16`), one temperature-1 sample per view (`S16`), canonical DFS (`D1`), 16-view DFS (`D16`), and DFS with eight-view likelihood rescoring (`D16R`).
- `submission`: run only the primary NVARC inference path: 16 views, DFS with the original 20% probability threshold, eight-view rescoring, `score_kgmon`, and two attempts.

A Kaggle competition rerun always overrides the visible flag to `submission`. The decoder never receives or enforces the target grid shape. Valid rectangular grids of any size from 1x1 through 30x30 are retained; shape mismatch is only an evaluation diagnostic.

## Kaggle inputs and runtime assumptions

Attach the ARC Prize 2026 competition data and the published NVARC model dataset, select four NVIDIA L4 GPUs, and keep Internet disabled. The notebook loads both inputs with `local_files_only=True`; it does not install packages or download artifacts.

The public evaluation run is intentionally broader than the competition submission run. Start with `EVALUATION_TASK_LIMIT = 4` as a smoke test, inspect coverage and timing, then set it back to `None` for all public tasks.

Inference structure and constants are adapted from [koushikrudra/failed-in-aimo](https://www.kaggle.com/code/koushikrudra/failed-in-aimo). The intentional differences are the removal of all test-time training and the addition of isolated evaluation variants and reproducible diagnostics.

In [ ]:
# User configuration: this is the only cell that normally needs editing.
from __future__ import annotations

import json
import os
import random
import time
from pathlib import Path

RUN_MODE = "evaluation"  # "evaluation" or "submission"
EVALUATION_TASK_LIMIT = None  # Use 4 for a short public-evaluation smoke test.

MODEL_PATH = "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"
COMPETITION_DATA_DIR = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"

NUM_GPUS = 4
EXPECTED_GPU_SUBSTRING = "L4"
MAX_SEQ_LENGTH = 8192
RANDOM_SEED = 42
SAMPLING_TEMPERATURE = 1.0
DFS_TOKEN_PROBABILITY_THRESHOLD = 0.20
DFS_CALL_TIME_LIMIT_SECONDS = 540
TASK_TIME_LIMIT_SECONDS = 1200
NOTEBOOK_TIME_LIMIT_SECONDS = 12 * 3600
SHUTDOWN_MARGIN_SECONDS = 600

assert RUN_MODE in {"evaluation", "submission"}
assert EVALUATION_TASK_LIMIT is None or EVALUATION_TASK_LIMIT > 0
assert SAMPLING_TEMPERATURE > 0
assert 0 < DFS_TOKEN_PROBABILITY_THRESHOLD < 1

def env_flag(name: str) -> bool:
    return os.getenv(name, "").strip().lower() in {"1", "true", "yes", "y"}

competition_rerun = env_flag("KAGGLE_IS_COMPETITION_RERUN")
effective_run_mode = "submission" if competition_rerun else RUN_MODE

data_dir = Path(COMPETITION_DATA_DIR)
challenge_path = data_dir / (
    "arc-agi_test_challenges.json"
    if effective_run_mode == "submission"
    else "arc-agi_evaluation_challenges.json"
)
solutions_path = (
    None
    if effective_run_mode == "submission"
    else data_dir / "arc-agi_evaluation_solutions.json"
)

run_id = time.strftime("%Y%m%d-%H%M%S") + f"-{effective_run_mode}"
run_dir = Path("/kaggle/working/nvarc_frozen_runs") / run_id
run_dir.mkdir(parents=True, exist_ok=False)

random.seed(RANDOM_SEED)
global_deadline = time.time() + NOTEBOOK_TIME_LIMIT_SECONDS - SHUTDOWN_MARGIN_SECONDS

config = {
    "requested_run_mode": RUN_MODE,
    "run_mode": effective_run_mode,
    "competition_rerun": competition_rerun,
    "evaluation_task_limit": EVALUATION_TASK_LIMIT,
    "model_path": MODEL_PATH,
    "competition_data_dir": str(data_dir),
    "challenge_path": str(challenge_path),
    "solutions_path": None if solutions_path is None else str(solutions_path),
    "run_id": run_id,
    "run_dir": str(run_dir),
    "num_gpus": NUM_GPUS,
    "expected_gpu_substring": EXPECTED_GPU_SUBSTRING,
    "max_seq_length": MAX_SEQ_LENGTH,
    "seed": RANDOM_SEED,
    "sampling_temperature": SAMPLING_TEMPERATURE,
    "dfs_token_probability_threshold": DFS_TOKEN_PROBABILITY_THRESHOLD,
    "dfs_call_time_limit_seconds": DFS_CALL_TIME_LIMIT_SECONDS,
    "task_time_limit_seconds": TASK_TIME_LIMIT_SECONDS,
    "global_deadline": global_deadline,
    "primary_selector": "D16R",
}

config_path = run_dir / "config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
config

In [ ]:
# Fail early on missing Kaggle inputs. No downloads or package installation occur here.
import importlib.metadata
import platform

assert Path(config["model_path"]).is_dir(), config["model_path"]
assert Path(config["challenge_path"]).is_file(), config["challenge_path"]
if config["solutions_path"] is not None:
    assert Path(config["solutions_path"]).is_file(), config["solutions_path"]

environment = {
    "python": platform.python_version(),
    "torch": importlib.metadata.version("torch"),
    "transformers": importlib.metadata.version("transformers"),
    "unsloth": importlib.metadata.version("unsloth"),
    "model_path": config["model_path"],
    "challenge_path": config["challenge_path"],
}
(run_dir / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
environment

In [ ]:
%%writefile nvarc_frozen.py
from __future__ import annotations

import bz2
import hashlib
import importlib.metadata
import json
import math
import os
import pickle
import random
import time
import traceback
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from transformers import AutoTokenizer


ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_start|>": 11,
    "user": 11,
    "assistant": 12,
    "<|padding|>": 13,
    "<|im_end|>": 15,
}
ARC_TOKENS = list(range(11)) + [15]
PAD_ID = 13
EOS_ID = 15


def stable_seed(*parts: object) -> int:
    payload = "|".join(map(str, parts)).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "big") % (2**31)


def hashable_grid(grid: np.ndarray) -> tuple[tuple[int, ...], ...]:
    return tuple(tuple(int(cell) for cell in row) for row in np.asarray(grid))


def convert_grid_to_string(grid: np.ndarray | list[list[int]]) -> str:
    return "\n".join("".join(str(int(cell)) for cell in row) for row in grid)


def is_valid_grid(grid: object) -> bool:
    if not isinstance(grid, np.ndarray) or grid.ndim != 2:
        return False
    if not all(1 <= size <= 30 for size in grid.shape):
        return False
    return bool(np.all((0 <= grid) & (grid <= 9)))


def shuffled(items: list[str]) -> list[str]:
    return np.random.permutation(items).tolist()


def permute_mod(array, descriptor: str, invert: bool = False):
    permutation = [int(char) for char in descriptor if char.isdigit()]
    assert sorted(permutation) == list(range(10))
    array = np.asarray(array)
    if array.ndim == 3:
        if not invert:
            permutation = np.argsort(permutation)
        return array[..., permutation]
    assert array.ndim == 2
    if invert:
        permutation = np.argsort(permutation)
    return np.asarray(permutation)[array]


def random_permutation_descriptor(_query) -> str:
    return "permute" + "".join(map(str, np.random.permutation(10).tolist()))


class QwenFormatter:
    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query: list[dict]) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply: list) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train: list[dict], last_is_challenge: bool = False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for example in train:
            grid_input = convert_grid_to_string(example["input"])
            grid_output = convert_grid_to_string(example["output"])
            text += (
                f"<|im_start|>user\n{grid_input}<|im_end|>"
                f"<|im_start|>assistant\n{grid_output}<|im_end|>"
            )
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self) -> int:
        reply = np.zeros((30, 30), dtype=int)
        return len(self.tokenizer.encode(self.fmt_reply([reply]))) + 1

    def convert_tokens_to_array(self, tokens: list[int]) -> np.ndarray | None:
        if not tokens or tokens[-1] != EOS_ID:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        lines = text.strip().splitlines()
        if not lines or len(lines) > 30:
            return None
        rows: list[list[int]] = []
        for line in lines:
            line = line.strip()
            if not line or len(line) > 30 or any(char not in "0123456789" for char in line):
                return None
            rows.append([int(char) for char in line])
        if len({len(row) for row in rows}) != 1:
            return None
        array = np.asarray(rows, dtype=np.int8)
        return array if is_valid_grid(array) else None


class ArcDataset:
    @staticmethod
    def forward_mod(array, key: str, use_perm: bool = True):
        if array is None:
            return array
        for operation in key.split(".")[1:]:
            if operation == "rot90":
                array = np.rot90(array)
            elif operation == "transpose":
                array = np.swapaxes(array, 0, 1)
            elif operation.startswith("permute"):
                array = permute_mod(array, operation, invert=False) if use_perm else array
            elif operation.startswith(("copy", "out", "ex", "run")):
                array = np.copy(array) if operation.startswith("copy") else array
            else:
                raise NotImplementedError(f"Unknown operation: {operation}")
        return array

    @staticmethod
    def invert_mod(array, key: str, inv_perm: bool = True):
        if array is None:
            return array
        for operation in key.split(".")[1:][::-1]:
            if operation == "rot90":
                array = np.rot90(array, k=3)
            elif operation == "transpose":
                array = np.swapaxes(array, 0, 1)
            elif operation.startswith("permute"):
                array = permute_mod(array, operation, invert=True) if inv_perm else array
            elif operation.startswith(("copy", "out", "ex", "run")):
                array = np.copy(array) if operation.startswith("copy") else array
            else:
                raise NotImplementedError(f"Unknown operation: {operation}")
        return array

    def __init__(self, queries: dict, replies: dict | None = None, keys=None, is_orig: bool = False):
        replies = {} if replies is None else replies
        if keys is not None:
            keys = [key for key in keys if key is not None]
        self.queries = queries if keys is None else {key: queries[key] for key in keys}
        self.replies = replies if keys is None else {key: replies[key] for key in keys if key in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries) if keys is None else list(keys)
        self.transposed_dataset = None

    @classmethod
    def from_file(cls, queries_file: str | Path, keys=None):
        with open(queries_file, encoding="utf-8") as handle:
            return cls(json.load(handle), keys=keys, is_orig=True)

    def load_replies(self, replies_file: str | Path):
        with open(replies_file, encoding="utf-8") as handle:
            parsed = json.load(handle)
        self.replies = {key: parsed[key] for key in self.keys}
        return self

    def change_keys(self, keys, keep_flags: bool = False):
        flags = {"is_orig": self.is_orig} if keep_flags else {}
        return self.__class__(self.queries, self.replies, keys, **flags)

    def split_multi_replies(self):
        key_indices = [
            (key, index)
            for key in self.keys
            for index in range(len(self.queries[key]["test"]))
        ]
        return self.__class__(
            keys=[f"{key}_{index}" for key, index in key_indices],
            queries={
                f"{key}_{index}": {
                    "train": self.queries[key]["train"],
                    "test": [self.queries[key]["test"][index]],
                }
                for key, index in key_indices
            },
            replies={
                f"{key}_{index}": [self.replies[key][index]]
                for key, index in key_indices
                if key in self.replies
            },
        )

    def shuffled(self):
        return self.__class__(self.queries, self.replies, shuffled(self.keys))

    @classmethod
    def append(cls, *datasets):
        return cls(
            queries={key: value for dataset in datasets for key, value in dataset.queries.items()},
            replies={key: value for dataset in datasets for key, value in dataset.replies.items()},
            keys=[key for dataset in datasets for key in dataset.keys],
        )

    def mod_single(self, mod_func, descriptor, index: int, keep_key: bool, inputs_only: bool):
        queries, replies, keys = {}, {}, []
        for old_key in self.keys:
            if descriptor is None:
                description = "copy{index}" if mod_func is np.copy else mod_func.__name__
            elif isinstance(descriptor, str):
                description = descriptor
            else:
                description = descriptor(self.queries[old_key])
            description = description.format(index=index)

            def transform(array, desc=description):
                value = mod_func(array) if descriptor is None else mod_func(array, desc)
                return np.asarray(value).tolist()

            new_key = old_key if keep_key else f"{old_key}.{'I' if inputs_only else ''}{description}"
            keys.append(new_key)
            queries[new_key] = {
                part: [
                    {
                        field: transform(array) if field == "input" or not inputs_only else array
                        for field, array in example.items()
                    }
                    for example in examples
                ]
                for part, examples in self.queries[old_key].items()
            }
            if old_key in self.replies:
                replies[new_key] = [transform(array) for array in self.replies[old_key]]
        return self.__class__(queries, replies, keys)

    def mod(
        self,
        mod_func,
        descriptor=None,
        n: int = 1,
        stack=None,
        keep: bool = False,
        keep_key: bool = False,
        shuffle: bool = False,
        join: bool = True,
        inputs_only: bool = False,
    ):
        assert not (keep and keep_key)
        current = self
        result = [current.shuffled() if shuffle else current] if keep else []
        if stack is None:
            stack = mod_func.__name__.startswith("rot")
        for index in range(n):
            source = current if stack else self
            current = source.mod_single(mod_func, descriptor, index, keep_key, inputs_only)
            result.append(current.shuffled() if shuffle else current)
        return self.__class__.append(*result) if join else result

    def shuffle_examples(self, permutation=None):
        new_keys, new_queries, new_replies = [], {}, {}
        for key in self.keys:
            count = len(self.queries[key]["train"])
            order = np.random.permutation(count) if permutation is None else np.asarray(permutation)
            separator = "-" if order.max(initial=0) > 9 else ""
            new_key = f"{key}.ex{separator.join(map(str, order.tolist()))}"
            new_keys.append(new_key)
            new_queries[new_key] = {
                part: np.asarray(examples, dtype=object)[order].tolist() if part == "train" else examples
                for part, examples in self.queries[key].items()
            }
            if key in self.replies:
                new_replies[new_key] = self.replies[key]
        return self.__class__(new_queries, new_replies, new_keys)

    def augment(self, n: int = 1, shuffle_keys: bool = False, seed: int = 42):
        np.random.seed(seed)
        dataset = self.mod(np.transpose, keep=True)
        dataset = dataset.mod(np.rot90, n=3, keep=True)
        dataset = dataset.mod(
            permute_mod,
            random_permutation_descriptor,
            n=n,
            shuffle=shuffle_keys,
            keep=False,
        )
        return dataset.shuffle_examples()

    def get(self, key: str, formatter: QwenFormatter) -> dict:
        train = formatter.fmt_train(self.queries[key]["train"])
        query = formatter.fmt_query(self.queries[key]["test"])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ""
        if reply:
            text = train + query + reply
        else:
            text = formatter.fmt_train(self.queries[key]["train"], last_is_challenge=True)
        return {"key": key, "train": train, "query": query, "reply": reply, "input": train + query, "text": text}

    def as_list(self, formatter: QwenFormatter) -> list[dict]:
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key: str, formatter: QwenFormatter, name: str) -> int:
        return len(formatter.tokenizer.encode(self.get(key, formatter)[name]))

    def cut_to_len(self, formatter: QwenFormatter, name: str, max_len: int):
        temporary = self.change_keys(self.keys)
        new_keys, new_queries, new_replies = [], {}, {}
        for original_key in self.keys:
            key = original_key
            reply = temporary.replies.get(key)
            while temporary.get_length(key, formatter, name) > max_len:
                query = temporary.queries[key]
                if not key.split(".")[-1].startswith("ex"):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_parts = key.split(".")
                assert key_parts[-1].startswith("ex")
                retained = key_parts[-1][3:]
                key = ".".join(key_parts[:-1] + [f"ex{retained}"])
                temporary.queries[key] = {
                    part: examples[1:] if part == "train" else examples
                    for part, examples in query.items()
                }
                if reply is not None:
                    temporary.replies[key] = reply
                if not temporary.queries[key]["train"] and temporary.get_length(key, formatter, name) > max_len:
                    raise ValueError(f"{original_key} exceeds {max_len} tokens even without demonstrations")
            new_keys.append(key)
            new_queries[key] = temporary.queries[key]
            if reply is not None:
                new_replies[key] = reply
        return self.__class__(new_queries, new_replies, new_keys)

    def get_submission(self, selected: dict | None = None) -> dict:
        assert self.is_orig, "Submission must be based on the original unsplit dataset"
        submission = {
            key: [
                {"attempt_1": [[0]], "attempt_2": [[0]]}
                for _ in range(len(self.queries[key]["test"]))
            ]
            for key in self.keys
        }
        if selected is not None:
            for split_key, guesses in selected.items():
                task_id, test_index = split_key.rsplit("_", 1)
                target = submission[task_id][int(test_index)]
                for index, guess in enumerate(guesses[:2]):
                    target[f"attempt_{index + 1}"] = np.asarray(guess).tolist()
        return submission


def verify_checkpoint_contract(model, tokenizer) -> dict:
    for token, expected_id in [(str(i), i) for i in range(10)]:
        actual = tokenizer.convert_tokens_to_ids(token)
        assert actual == expected_id, (token, actual, expected_id)
    assert tokenizer.convert_tokens_to_ids("Ċ") == 10
    assert tokenizer.convert_tokens_to_ids("<|im_end|>") == EOS_ID
    embedding_rows = int(model.get_input_embeddings().weight.shape[0])
    output_rows = int(model.get_output_embeddings().weight.shape[0])
    assert embedding_rows == output_rows == 16, (embedding_rows, output_rows)
    return {
        "embedding_rows": embedding_rows,
        "output_rows": output_rows,
        "parameter_dtypes": sorted({str(parameter.dtype) for parameter in model.parameters()}),
    }


def load_frozen_model(config: dict):
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=config["model_path"],
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=config["max_seq_length"],
        dtype=torch.bfloat16,
    )
    model = FastLanguageModel.for_inference(model)
    model.requires_grad_(False)
    model.eval()
    assert not any(parameter.requires_grad for parameter in model.parameters())
    contract = verify_checkpoint_contract(model, tokenizer)
    return model, tokenizer, contract


@torch.inference_mode()
def turbo_dfs(
    model,
    logits,
    max_new_tokens: int,
    max_score: float,
    scores: list[float],
    position: int,
    cache,
    search_started: float,
    call_time_limit: float,
    deadline: float,
) -> dict[int, list[tuple[float, list[int]]]]:
    batch_size = logits.size(0)
    nll = torch.tensor(scores, dtype=torch.float32).view(batch_size, 1) - logits.float().cpu().log_softmax(-1)
    suffixes = defaultdict(list)
    candidates: dict[int, list[tuple[float, int]]] = {}

    for batch_index in range(batch_size):
        candidates[batch_index] = []
        for token in ARC_TOKENS:
            score = nll[batch_index, token].item()
            if score < max_score:
                if token == EOS_ID:
                    suffixes[batch_index].append((score, [token]))
                elif max_new_tokens > 1:
                    candidates[batch_index].append((score, token))
        candidates[batch_index].sort(key=lambda item: item[0])

    device = next(model.parameters()).device
    while time.time() - search_started < call_time_limit and time.time() < deadline:
        batch_tokens, batch_scores = [], []
        alive = 0
        for batch_index in range(batch_size):
            if not candidates[batch_index]:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000.0)
            else:
                score, token = candidates[batch_index].pop(0)
                batch_tokens.append(token)
                batch_scores.append(score)
                alive += 1
        if alive == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((batch_size, 1), position, device=device, dtype=torch.long),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )
        next_suffixes = turbo_dfs(
            model=model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens - 1,
            max_score=max_score,
            scores=batch_scores,
            position=position + 1,
            cache=outputs.past_key_values,
            search_started=search_started,
            call_time_limit=call_time_limit,
            deadline=deadline,
        )
        for batch_index, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffixes[batch_index].append((score, [batch_tokens[batch_index], *suffix_tokens]))
    return suffixes


@torch.inference_mode()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, call_time_limit, deadline):
    lengths = {len(tokens) for tokens in prefix_tokens}
    assert len(lengths) == 1, f"DFS batch has unequal prompt lengths: {sorted(lengths)}"
    device = next(model.parameters()).device
    input_ids = torch.tensor(prefix_tokens, device=device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model=model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        position=input_ids.size(1),
        cache=outputs.past_key_values,
        search_started=time.time(),
        call_time_limit=call_time_limit,
        deadline=deadline,
    )
    return {
        batch_index: sorted(beams, key=lambda item: item[0])
        for batch_index, beams in suffixes.items()
    }


@torch.inference_mode()
def autoregressive_decode(model, prefix_tokens, max_new_tokens, do_sample, temperature, seed):
    device = next(model.parameters()).device
    input_ids = torch.tensor([prefix_tokens], device=device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    cache = outputs.past_key_values
    logits = outputs.logits[0, -1]
    generated: list[int] = []
    nll = 0.0
    generator = torch.Generator(device=device)
    generator.manual_seed(seed)

    for position_offset in range(max_new_tokens):
        full_log_probs = logits.float().log_softmax(-1)
        allowed_logits = logits[ARC_TOKENS].float() / temperature
        if do_sample:
            relative_index = torch.multinomial(
                allowed_logits.softmax(-1),
                num_samples=1,
                generator=generator,
            ).item()
        else:
            relative_index = int(torch.argmax(allowed_logits).item())
        token = ARC_TOKENS[relative_index]
        nll -= float(full_log_probs[token].item())
        generated.append(token)
        if token == EOS_ID:
            break
        outputs = model(
            input_ids=torch.tensor([[token]], device=device, dtype=torch.long),
            position_ids=torch.tensor([[len(prefix_tokens) + position_offset]], device=device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )
        cache = outputs.past_key_values
        logits = outputs.logits[0, -1]
    return nll, generated


@torch.inference_mode()
def calc_scores(queries, answers, tokenizer, model) -> list[float]:
    query_tokens = [tokenizer.encode(query) for query in queries]
    answer_tokens = [tokenizer.encode(answer) for answer in answers]
    combined = [query + answer for query, answer in zip(query_tokens, answer_tokens)]
    max_length = max(map(len, combined))
    padded = [tokens + [PAD_ID] * (max_length - len(tokens)) for tokens in combined]
    device = next(model.parameters()).device
    input_ids = torch.tensor(padded, device=device, dtype=torch.long)
    logits = model(input_ids=input_ids, return_dict=True, use_cache=False).logits.float().cpu().log_softmax(-1)
    scores = []
    for sample_logits, query, answer in zip(logits, query_tokens, answer_tokens):
        start = len(query) - 1
        answer_logits = sample_logits[start : start + len(answer)]
        indices = torch.arange(len(answer))
        scores.append(float(-answer_logits[indices, answer].sum().item()))
    return scores


def batch_keys_by_prompt_length(dataset: ArcDataset, formatter: QwenFormatter, max_batch_size: int = 4):
    grouped = defaultdict(list)
    for key in sorted(dataset.keys):
        token_count = len(formatter.tokenizer.encode(dataset.get(key, formatter)["input"]))
        grouped[token_count].append(key)
    batches = []
    for token_count in sorted(grouped):
        keys = grouped[token_count]
        batches.extend(keys[index : index + max_batch_size] for index in range(0, len(keys), max_batch_size))
    return batches


def decoded_record(formatter, split_dataset, subkey, score, tokens, method):
    array = formatter.convert_tokens_to_array(tokens)
    if array is None:
        return None
    solution = np.asarray(split_dataset.invert_mod(array, subkey, inv_perm=True), dtype=np.int8)
    assert is_valid_grid(solution)
    return {
        "base_key": subkey.split(".")[0],
        "source_key": subkey,
        "method": method,
        "beam_score": float(score),
        "score_aug": None,
        "solution": solution,
    }


def deduplicated_groups(records: list[dict]):
    groups = defaultdict(list)
    for record in records:
        groups[hashable_grid(record["solution"])].append(record)
    return groups


def select_frequency_nll(records: list[dict]) -> list[np.ndarray]:
    ranked = []
    for group in deduplicated_groups(records).values():
        mean_nll = float(np.mean([record["beam_score"] for record in group]))
        ranked.append(((len(group), -mean_nll), group[0]["solution"]))
    ranked.sort(key=lambda item: item[0], reverse=True)
    return [solution for _, solution in ranked]


def select_lowest_nll(records: list[dict]) -> list[np.ndarray]:
    ranked = []
    for group in deduplicated_groups(records).values():
        best_nll = min(record["beam_score"] for record in group)
        ranked.append((best_nll, group[0]["solution"]))
    ranked.sort(key=lambda item: item[0])
    return [solution for _, solution in ranked]


def select_kgmon(records: list[dict]) -> list[np.ndarray]:
    ranked = []
    for group in deduplicated_groups(records).values():
        assert all(record["score_aug"] is not None for record in group)
        frequency = len(group)
        rescore_nll = float(np.mean([np.mean(record["score_aug"]) for record in group]))
        ranked.append((frequency - rescore_nll, group[0]["solution"]))
    ranked.sort(key=lambda item: item[0], reverse=True)
    return [solution for _, solution in ranked]


def select_full_probmul_3(records: list[dict], baseline: float = 3.0) -> list[np.ndarray]:
    ranked = []
    for group in deduplicated_groups(records).values():
        assert all(record["score_aug"] is not None for record in group)
        inference_score = sum(baseline - record["beam_score"] for record in group)
        augmentation_score = float(
            np.mean([
                sum(baseline - score for score in record["score_aug"])
                for record in group
            ])
        )
        ranked.append((inference_score + augmentation_score, group[0]["solution"]))
    ranked.sort(key=lambda item: item[0], reverse=True)
    return [solution for _, solution in ranked]


def append_decoded(records, stats, variant, formatter, split_dataset, subkey, score, tokens):
    stats[variant]["generated"] += 1
    record = decoded_record(formatter, split_dataset, subkey, score, tokens, variant)
    if record is None:
        stats[variant]["invalid"] += 1
    else:
        stats[variant]["valid"] += 1
        records[variant].append(record)


def run_dfs_dataset(model, tokenizer, formatter, dataset, split_dataset, config, deadline, variant):
    records = []
    stats = {"generated": 0, "valid": 0, "invalid": 0, "batches_completed": 0}
    max_score = -math.log(config["dfs_token_probability_threshold"])
    max_new_tokens = formatter.max_new_tokens()
    for subkeys in batch_keys_by_prompt_length(dataset, formatter):
        if time.time() >= deadline:
            break
        prefixes = [tokenizer.encode(dataset.get(subkey, formatter)["input"]) for subkey in subkeys]
        result = inference_turbo_dfs(
            model,
            prefixes,
            max_new_tokens,
            max_score,
            config["dfs_call_time_limit_seconds"],
            deadline,
        )
        stats["batches_completed"] += 1
        for batch_index, beams in result.items():
            subkey = subkeys[batch_index]
            for score, tokens in beams:
                stats["generated"] += 1
                record = decoded_record(formatter, split_dataset, subkey, score, tokens, variant)
                if record is None:
                    stats["invalid"] += 1
                else:
                    stats["valid"] += 1
                    records.append(record)
    return records, stats


def run_single_path(model, tokenizer, formatter, dataset, split_dataset, config, deadline, variant, do_sample):
    records = []
    stats = {"generated": 0, "valid": 0, "invalid": 0, "views_completed": 0}
    max_new_tokens = formatter.max_new_tokens()
    for subkey in sorted(dataset.keys):
        if time.time() >= deadline:
            break
        prefix = tokenizer.encode(dataset.get(subkey, formatter)["input"])
        seed = stable_seed(config["seed"], variant, subkey)
        score, tokens = autoregressive_decode(
            model,
            prefix,
            max_new_tokens,
            do_sample=do_sample,
            temperature=config["sampling_temperature"],
            seed=seed,
        )
        stats["views_completed"] += 1
        stats["generated"] += 1
        record = decoded_record(formatter, split_dataset, subkey, score, tokens, variant)
        if record is None:
            stats["invalid"] += 1
        else:
            stats["valid"] += 1
            records.append(record)
    return records, stats


def rescore_d16_records(
    model,
    tokenizer,
    formatter,
    split_dataset,
    records,
    config,
    deadline,
    cache=None,
):
    cache = {} if cache is None else cache
    rescored = []
    for record in records:
        cache_key = (record["base_key"], hashable_grid(record["solution"]))
        if cache_key not in cache:
            if time.time() >= deadline:
                break
            base_key = record["base_key"]
            candidate_dataset = ArcDataset(
                keys=[base_key],
                queries={base_key: split_dataset.queries[base_key]},
                replies={base_key: [record["solution"].tolist()]},
            ).augment(n=1, seed=stable_seed(config["seed"], "rescore", base_key))
            candidate_dataset = candidate_dataset.cut_to_len(
                formatter,
                "input",
                config["max_seq_length"] - formatter.max_new_tokens(),
            )
            queries, answers = [], []
            for sample in candidate_dataset.as_list(formatter):
                queries.append(sample["input"])
                answers.append(sample["reply"])
            scores = []
            for start in range(0, len(queries), 4):
                scores.extend(calc_scores(
                    queries[start : start + 4],
                    answers[start : start + 4],
                    tokenizer,
                    model,
                ))
            assert len(scores) == 8
            cache[cache_key] = scores
        rescored_record = dict(record)
        rescored_record["score_aug"] = cache[cache_key]
        rescored.append(rescored_record)
    return rescored, {"unique_candidates_rescored": len(cache), "records_rescored": len(rescored)}


def run_d16_with_rescoring(
    model,
    tokenizer,
    formatter,
    views,
    split_dataset,
    config,
    deadline,
):
    d16_records = []
    d16r_records = []
    d16_stats = {"generated": 0, "valid": 0, "invalid": 0, "batches_completed": 0}
    rescore_cache = {}
    decode_seconds = 0.0
    rescore_seconds = 0.0
    max_score = -math.log(config["dfs_token_probability_threshold"])
    max_new_tokens = formatter.max_new_tokens()

    for subkeys in batch_keys_by_prompt_length(views, formatter):
        if time.time() >= deadline:
            break
        stage_started = time.time()
        prefixes = [tokenizer.encode(views.get(subkey, formatter)["input"]) for subkey in subkeys]
        result = inference_turbo_dfs(
            model,
            prefixes,
            max_new_tokens,
            max_score,
            config["dfs_call_time_limit_seconds"],
            deadline,
        )
        decode_seconds += time.time() - stage_started
        d16_stats["batches_completed"] += 1

        batch_records = []
        for batch_index, beams in result.items():
            subkey = subkeys[batch_index]
            for score, tokens in beams:
                d16_stats["generated"] += 1
                record = decoded_record(formatter, split_dataset, subkey, score, tokens, "D16")
                if record is None:
                    d16_stats["invalid"] += 1
                else:
                    d16_stats["valid"] += 1
                    batch_records.append(record)
        d16_records.extend(batch_records)

        # Match the reference pipeline: score each batch before starting the next DFS batch.
        stage_started = time.time()
        batch_rescored, _ = rescore_d16_records(
            model,
            tokenizer,
            formatter,
            split_dataset,
            batch_records,
            config,
            deadline,
            cache=rescore_cache,
        )
        d16r_records.extend(batch_rescored)
        rescore_seconds += time.time() - stage_started

    rescore_stats = {
        "unique_candidates_rescored": len(rescore_cache),
        "records_rescored": len(d16r_records),
    }
    runtimes = {"D16_decode": decode_seconds, "D16_rescore": rescore_seconds}
    return d16_records, d16r_records, d16_stats, rescore_stats, runtimes


def records_by_base_key(records: list[dict]) -> dict[str, list[dict]]:
    grouped = defaultdict(list)
    for record in records:
        grouped[record["base_key"]].append(record)
    return grouped


def select_all(candidate_sets: dict[str, list[dict]]) -> dict[str, dict[str, list[np.ndarray]]]:
    selected = defaultdict(dict)
    selectors = {
        "G0": ("G0", select_lowest_nll),
        "G16": ("G16", select_frequency_nll),
        "S16": ("S16", select_frequency_nll),
        "D1": ("D1", select_lowest_nll),
        "D16": ("D16", select_frequency_nll),
        "D16R": ("D16R", select_kgmon),
        "D16R_probmul": ("D16R", select_full_probmul_3),
    }
    for output_name, (candidate_name, selector) in selectors.items():
        for base_key, records in records_by_base_key(candidate_sets.get(candidate_name, [])).items():
            selected[output_name][base_key] = selector(records)
    return dict(selected)


def solve_task(model, tokenizer, formatter, arc_dataset, task_id: str, config: dict) -> dict:
    task_started = time.time()
    deadline = min(
        config["global_deadline"],
        task_started + config["task_time_limit_seconds"],
    )
    puzzle = arc_dataset.change_keys([task_id])
    split_dataset = puzzle.split_multi_replies()
    canonical = split_dataset.cut_to_len(
        formatter,
        "input",
        config["max_seq_length"] - formatter.max_new_tokens(),
    )
    views = split_dataset.augment(n=2, seed=2).cut_to_len(
        formatter,
        "input",
        config["max_seq_length"] - formatter.max_new_tokens(),
    )
    candidate_sets: dict[str, list[dict]] = {}
    generation_stats: dict[str, dict] = {}
    runtime_seconds: dict[str, float] = {}

    (
        d16_records,
        d16r_records,
        generation_stats["D16"],
        generation_stats["D16R"],
        primary_runtimes,
    ) = run_d16_with_rescoring(
        model, tokenizer, formatter, views, split_dataset, config, deadline
    )
    candidate_sets["D16"] = d16_records
    candidate_sets["D16R"] = d16r_records
    runtime_seconds.update(primary_runtimes)

    if config["run_mode"] == "evaluation":
        for variant, dataset, do_sample in [
            ("G0", canonical, False),
            ("G16", views, False),
            ("S16", views, True),
        ]:
            stage_started = time.time()
            records, stats = run_single_path(
                model, tokenizer, formatter, dataset, split_dataset, config, deadline, variant, do_sample
            )
            candidate_sets[variant] = records
            generation_stats[variant] = stats
            runtime_seconds[f"{variant}_decode"] = time.time() - stage_started

        stage_started = time.time()
        d1_records, generation_stats["D1"] = run_dfs_dataset(
            model, tokenizer, formatter, canonical, split_dataset, config, deadline, "D1"
        )
        candidate_sets["D1"] = d1_records
        runtime_seconds["D1_decode"] = time.time() - stage_started

    selections = select_all(candidate_sets)
    expected_outputs = len(split_dataset.keys)
    result = {
        "task_id": task_id,
        "candidate_sets": candidate_sets,
        "selections": selections,
        "generation_stats": generation_stats,
        "runtime_seconds": runtime_seconds,
        "expected_outputs": expected_outputs,
        "elapsed_seconds": time.time() - task_started,
        "deadline_reached": time.time() >= deadline,
    }
    return result


def write_pickle_bz2(path: str | Path, value) -> None:
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with bz2.BZ2File(temporary, "wb") as handle:
        pickle.dump(value, handle, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(temporary, path)


def worker(rank: int, queue, config_path: str):
    with open(config_path, encoding="utf-8") as handle:
        config = json.load(handle)
    run_dir = Path(config["run_dir"])
    output_dir = run_dir / "task_results"
    error_dir = run_dir / "errors"
    output_dir.mkdir(parents=True, exist_ok=True)
    error_dir.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(config["seed"] + rank)
    np.random.seed(config["seed"] + rank)
    random.seed(config["seed"] + rank)
    model, tokenizer, contract = load_frozen_model(config)
    formatter = QwenFormatter(tokenizer)
    dataset = ArcDataset.from_file(config["challenge_path"])

    manifest = {
        "rank": rank,
        "gpu": torch.cuda.get_device_name(0),
        "torch": importlib.metadata.version("torch"),
        "transformers": importlib.metadata.version("transformers"),
        "unsloth": importlib.metadata.version("unsloth"),
        "checkpoint_contract": contract,
        "all_parameters_frozen": not any(parameter.requires_grad for parameter in model.parameters()),
    }
    (run_dir / f"worker_{rank}_manifest.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )

    while True:
        if time.time() >= config["global_deadline"]:
            break
        task_id = queue.get()
        if task_id is None:
            break
        try:
            result = solve_task(model, tokenizer, formatter, dataset, task_id, config)
            write_pickle_bz2(output_dir / f"{task_id}.pbz2", result)
            print(f"[GPU {rank}] {task_id}: {result['elapsed_seconds']:.1f}s")
        except Exception as error:
            failure = {
                "task_id": task_id,
                "rank": rank,
                "error": repr(error),
                "traceback": traceback.format_exc(),
            }
            (error_dir / f"{task_id}.json").write_text(
                json.dumps(failure, indent=2), encoding="utf-8"
            )
            print(f"[GPU {rank}] ERROR {task_id}: {error!r}")


def load_task_results(run_dir: str | Path) -> dict[str, dict]:
    results = {}
    for path in sorted((Path(run_dir) / "task_results").glob("*.pbz2")):
        with bz2.BZ2File(path, "rb") as handle:
            result = pickle.load(handle)
        results[result["task_id"]] = result
    return results


def merge_selections(task_results: dict[str, dict]) -> dict[str, dict[str, list[np.ndarray]]]:
    merged = defaultdict(dict)
    for result in task_results.values():
        for variant, by_output in result["selections"].items():
            merged[variant].update(by_output)
    return dict(merged)


def candidate_source_for_variant(variant: str) -> str:
    return "D16R" if variant in {"D16R", "D16R_probmul"} else variant


def evaluate_variant(original, task_ids, task_results, selected, variant: str) -> dict:
    ranked = selected.get(variant, {})
    candidate_source = candidate_source_for_variant(variant)
    candidate_records = defaultdict(list)
    generated = valid = invalid = 0
    runtime = defaultdict(float)
    deadline_tasks = 0
    for task_id, result in task_results.items():
        records = result["candidate_sets"].get(candidate_source, [])
        for record in records:
            candidate_records[record["base_key"]].append(record)
        stats_source = "D16" if candidate_source == "D16R" else candidate_source
        stats = result["generation_stats"].get(stats_source, {})
        generated += stats.get("generated", 0)
        valid += stats.get("valid", 0)
        invalid += stats.get("invalid", 0)
        deadline_tasks += int(result["deadline_reached"])
        for name, seconds in result["runtime_seconds"].items():
            runtime[name] += seconds

    pass1_weighted = pass2_weighted = 0.0
    oracle_weighted = {1: 0.0, 2: 0.0, 5: 0.0, 10: 0.0, "all": 0.0}
    correct_outputs_pass2 = set()
    full_task_solved = set()
    wrong_shape_unique = 0
    unique_candidates = 0
    covered_outputs = 0
    fully_covered_tasks = 0

    for task_id in task_ids:
        task_all_correct = True
        task_all_covered = True
        output_count = len(original.replies[task_id])
        weight = 1.0 / output_count
        for index, target in enumerate(original.replies[task_id]):
            base_key = f"{task_id}_{index}"
            target_array = np.asarray(target)
            predictions = ranked.get(base_key, [])
            covered_outputs += int(bool(predictions))
            task_all_covered &= bool(predictions)
            if predictions and np.array_equal(predictions[0], target_array):
                pass1_weighted += weight
            if any(np.array_equal(prediction, target_array) for prediction in predictions[:2]):
                pass2_weighted += weight
                correct_outputs_pass2.add(base_key)
            else:
                task_all_correct = False

            unique = deduplicated_groups(candidate_records.get(base_key, []))
            unique_candidates += len(unique)
            for group in unique.values():
                if tuple(group[0]["solution"].shape) != tuple(target_array.shape):
                    wrong_shape_unique += 1
            for k in [1, 2, 5, 10]:
                if any(np.array_equal(prediction, target_array) for prediction in predictions[:k]):
                    oracle_weighted[k] += weight
            if any(np.array_equal(prediction, target_array) for prediction in predictions):
                oracle_weighted["all"] += weight
        if task_all_correct:
            full_task_solved.add(task_id)
        fully_covered_tasks += int(task_all_covered)

    task_count = len(task_ids)
    assert task_count > 0
    return {
        "pass_at_1": pass1_weighted / task_count,
        "pass_at_2": pass2_weighted / task_count,
        "candidate_oracle_at_k": {
            str(k): score / task_count for k, score in oracle_weighted.items()
        },
        "generated_sequences": generated,
        "valid_sequences": valid,
        "invalid_sequences": invalid,
        "valid_rate": valid / generated if generated else None,
        "unique_candidates": unique_candidates,
        "wrong_shape_unique_candidates": wrong_shape_unique,
        "wrong_shape_candidates_were_filtered": False,
        "outputs_with_pass_at_2": sorted(correct_outputs_pass2),
        "fully_solved_tasks_at_2": sorted(full_task_solved),
        "covered_outputs": covered_outputs,
        "fully_covered_tasks": fully_covered_tasks,
        "deadline_reached_task_files": deadline_tasks,
        "runtime_seconds": dict(runtime),
    }


def pairwise_solved_overlap(metrics: dict[str, dict]) -> dict:
    variants = sorted(metrics)
    overlap = {}
    for left_index, left in enumerate(variants):
        left_set = set(metrics[left]["outputs_with_pass_at_2"])
        for right in variants[left_index + 1 :]:
            right_set = set(metrics[right]["outputs_with_pass_at_2"])
            union = left_set | right_set
            overlap[f"{left}__{right}"] = {
                "intersection": len(left_set & right_set),
                "union": len(union),
                "jaccard": len(left_set & right_set) / len(union) if union else None,
            }
    return overlap


def aggregate_run(config_path: str | Path) -> dict:
    config_path = Path(config_path)
    with config_path.open(encoding="utf-8") as handle:
        config = json.load(handle)
    run_dir = Path(config["run_dir"])
    original = ArcDataset.from_file(config["challenge_path"])
    task_results = load_task_results(run_dir)
    requested_task_ids = list(original.keys)
    if config["run_mode"] == "evaluation" and config["evaluation_task_limit"] is not None:
        requested_task_ids = requested_task_ids[: config["evaluation_task_limit"]]
    selected = merge_selections(task_results)
    primary = selected.get(config["primary_selector"], {})
    submission = original.get_submission(primary)

    submission_path = Path("/kaggle/working/submission.json")
    submission_path.write_text(json.dumps(submission), encoding="utf-8")
    (run_dir / "submission.json").write_text(json.dumps(submission), encoding="utf-8")
    write_pickle_bz2(run_dir / "candidate_data.pbz2", task_results)

    error_files = sorted(path.name for path in (run_dir / "errors").glob("*.json"))
    summary = {
        "run_id": config["run_id"],
        "run_mode": config["run_mode"],
        "requested_tasks": len(requested_task_ids),
        "completed_task_files": len(task_results),
        "error_files": error_files,
        "submission_path": str(submission_path),
        "candidate_data_path": str(run_dir / "candidate_data.pbz2"),
    }

    if config["run_mode"] == "evaluation":
        original.load_replies(config["solutions_path"])
        variants = ["G0", "G16", "S16", "D1", "D16", "D16R", "D16R_probmul"]
        metrics = {
            variant: evaluate_variant(original, requested_task_ids, task_results, selected, variant)
            for variant in variants
        }
        evaluation_results = {
            "config": config,
            "coverage": {
                "requested_tasks": len(requested_task_ids),
                "completed_task_files": len(task_results),
                "missing_task_ids": sorted(set(requested_task_ids) - set(task_results)),
                "error_files": error_files,
            },
            "metrics": metrics,
            "solved_output_overlap": pairwise_solved_overlap(metrics),
        }
        evaluation_path = run_dir / "evaluation_results.json"
        evaluation_path.write_text(
            json.dumps(evaluation_results, indent=2), encoding="utf-8"
        )
        summary["evaluation_results_path"] = str(evaluation_path)
        summary["metrics"] = {
            variant: {
                "pass_at_1": values["pass_at_1"],
                "pass_at_2": values["pass_at_2"],
                "oracle_all": values["candidate_oracle_at_k"]["all"],
                "valid_rate": values["valid_rate"],
            }
            for variant, values in metrics.items()
        }

    (run_dir / "run_summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    return summary

In [ ]:
%%writefile nvarc_runner.py
from __future__ import annotations

import argparse
import json
import os
import time
from pathlib import Path

import torch
import torch.multiprocessing as mp


def local_worker(rank: int, queue, config_path: str):
    with open(config_path, encoding="utf-8") as handle:
        config = json.load(handle)
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
    os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
    os.environ["OMP_NUM_THREADS"] = "12"
    torch.set_default_device("cpu")

    marker_dir = Path(config["run_dir"]) / "worker_markers"
    marker_dir.mkdir(parents=True, exist_ok=True)
    if rank > 0:
        previous_marker = marker_dir / f"worker_{rank - 1}.ready"
        while not previous_marker.exists():
            if time.time() >= config["global_deadline"]:
                raise TimeoutError(f"Timed out waiting for {previous_marker}")
            time.sleep(2)

    # Import Unsloth serially because its runtime patching is not process-safe.
    from unsloth import FastLanguageModel  # noqa: F401
    (marker_dir / f"worker_{rank}.ready").write_text("ready", encoding="utf-8")

    from nvarc_frozen import worker
    worker(rank, queue, config_path)


def main(config_path: str):
    with open(config_path, encoding="utf-8") as handle:
        config = json.load(handle)
    assert torch.cuda.is_available(), "CUDA is required"
    assert torch.cuda.device_count() >= config["num_gpus"], (
        torch.cuda.device_count(), config["num_gpus"]
    )
    gpu_names = [torch.cuda.get_device_name(index) for index in range(config["num_gpus"])]
    expected = config.get("expected_gpu_substring")
    if expected:
        assert all(expected in name for name in gpu_names), gpu_names

    with open(config["challenge_path"], encoding="utf-8") as handle:
        task_ids = sorted(json.load(handle))
    if config["run_mode"] == "evaluation" and config["evaluation_task_limit"] is not None:
        task_ids = task_ids[: config["evaluation_task_limit"]]

    manager = mp.Manager()
    queue = manager.Queue()
    for task_id in task_ids:
        queue.put(task_id)
    for _ in range(config["num_gpus"]):
        queue.put(None)

    print(f"Running {len(task_ids)} tasks on {config['num_gpus']} GPUs: {gpu_names}")
    mp.spawn(
        local_worker,
        args=(queue, config_path),
        nprocs=config["num_gpus"],
        join=True,
    )


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    arguments = parser.parse_args()
    main(arguments.config)

## Run inference

The primary `D16R` path runs first for every task. In evaluation mode, the extra greedy, sampling, and canonical-DFS variants use whatever remains of the original 1,200-second per-task budget. Coverage fields make any timeout-induced incompleteness explicit.

In [ ]:
import subprocess
import sys

command = [sys.executable, "nvarc_runner.py", "--config", str(config_path)]
print(" ".join(command))
subprocess.run(command, check=True)

## Aggregate candidates, metrics, and submission

Aggregation never uses the public labels for decoding or candidate filtering. Labels are loaded only here, after inference, to calculate exact-grid metrics and diagnostics. A wrong-shape candidate remains in its original rank and is counted as incorrect.

In [ ]:
from nvarc_frozen import aggregate_run

summary = aggregate_run(config_path)
summary

In [ ]:
# Compact comparison table for public evaluation; skipped in submission mode.
if "metrics" in summary:
    import pandas as pd

    comparison = pd.DataFrame(summary["metrics"]).T
    display(comparison.sort_index())
else:
    print(f"Submission written to {summary['submission_path']}")

## Produced artifacts

- `/kaggle/working/submission.json`: top-two `D16R`/`score_kgmon` attempts in Kaggle format.
- `<run_dir>/evaluation_results.json`: public exact-grid metrics, candidate oracle values, validity/shape diagnostics, coverage, timing, and solved-output overlap.
- `<run_dir>/candidate_data.pbz2`: compressed raw candidates and their scores for offline selector analysis.
- `<run_dir>/worker_*_manifest.json`: hardware, package versions, checkpoint-interface checks, and frozen-parameter confirmation.

The full GPU run is expected to take hours and must be measured on Kaggle. Do not interpret metrics from a task-limited smoke run as a full public-set result.